# Аналіз сделок з кількома рахунками

Виведення результатів з `result.csv`.

In [ ]:
import pandas as pd

df = pd.read_csv('result.csv', encoding='utf-8-sig')

# 1С зберігає дати зі зсувом +2000 років (рік 4024 = реальний 2024).
# Знімаємо 2000 на рівні рядка, потім парсимо — інакше pandas падає в NaT.
def fix_year(s):
    if pd.isna(s):
        return s
    return f'{int(s[:4]) - 2000:04d}{s[4:]}'

for col in ['ДатаСделки', 'ДатаСчета']:
    df[col] = pd.to_datetime(df[col].map(fix_year), format='%Y-%m-%d %H:%M:%S', errors='coerce')

print(f'Рядків: {len(df)}')
print(f'Унікальних сделок: {df["НомерСделки"].nunique()}')
print(f'Діапазон дат: {df["ДатаСделки"].min()} … {df["ДатаСделки"].max()}')
df.head(20)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## ТОП-20 сделок за кількістю рахунків

In [ ]:
top = (df.groupby(['НомерСделки', 'ДатаСделки', 'КолвоСчетов'], as_index=False)
         .agg(СуммаПоСчетам=('Сумма', 'sum'))
         .sort_values('КолвоСчетов', ascending=False)
         .head(20))
top

## Усі рядки

In [ ]:
pd.set_option('display.max_rows', None)
df

## Крок 2. Угоди, де рахунки виставлені в різні місяці

In [ ]:
# Місяць рахунку (рік-місяць)
df['МесяцСчета'] = df['ДатаСчета'].dt.to_period('M')

# Залишаємо тільки угоди, у яких рахунки потрапляють у >1 різний місяць.
# КолвоСчетов вже > 1 згідно SQL, тож додаткова перевірка не потрібна.
сделки_разные_месяцы = (
    df.groupby('НомерСделки')['МесяцСчета']
      .nunique()
      .loc[lambda s: s > 1]
      .index
)

df_разные_месяцы = (
    df[df['НомерСделки'].isin(сделки_разные_месяцы)]
      .sort_values(['НомерСделки', 'ДатаСчета'])
      .reset_index(drop=True)
)

print(f'Угод з рахунками в різні місяці: {df_разные_месяцы["НомерСделки"].nunique()}')
print(f'Рядків (рахунків): {len(df_разные_месяцы)}')
df_разные_месяцы

## Крок 3. Кількість таких угод з початку 2026 року

Враховується дата угоди (`ДатаСделки`) >= 2026-01-01.

In [ ]:
сделки_2026 = (
    df_разные_месяцы[df_разные_месяцы['ДатаСделки'] >= '2026-01-01']
      .groupby(['НомерСделки', 'ДатаСделки'], as_index=False)
      .agg(КолвоСчетов=('НомерСчета', 'count'),
           КолвоМесяцев=('МесяцСчета', 'nunique'),
           СуммаПоСчетам=('Сумма', 'sum'))
      .sort_values('ДатаСделки')
)

print(f'Угод з різними місяцями рахунків з 2026-01-01: {len(сделки_2026)}')
сделки_2026